In [7]:
# ==============================================================================
# 1. IMPORTS & GLOBAL CONFIGURATION
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
pd.core.strings.__dict__['StringMethods'] = pd.core.strings.accessor.StringMethods
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

# Format pandas output for readability
pd.set_option('display.max_columns', None)
pd.reset_option('display.float_format')
# ==============================================================================
# 2. DATA LOADING & EXPLORATORY DATA ANALYSIS (EDA)
# ==============================================================================
df = pd.read_csv('Melbourne_housing_FULL.csv')

print("--- First 5 rows of the dataset ---")
display(df.head())

print("\n--- Dataset Info & Data Types ---")
df.info()

print("\n--- Descriptive Statistics for Numerical Features ---")
display(df.describe())

print("\n--- Missing Values Count per Column ---")
display(df.isnull().sum()[df.isnull().sum() > 0])

# ==============================================================================
# 3. ANOMALY & OUTLIER FILTERING (PRE-SPLIT)
# ==============================================================================
df_clean = df.copy()

# 1. Drop rows where target variable (Price) is missing
df_clean = df_clean.dropna(subset=['Price'])

# 2. Filter out explicit physical errors without dropping NaNs
# Filter Landsize outliers
df_clean = df_clean[(df_clean['Landsize'].isnull()) | (df_clean['Landsize'] <= 5000)]

# Filter BuildingArea outliers
df_clean = df_clean[(df_clean['BuildingArea'].isnull()) | (df_clean['BuildingArea'] <= 600)]

# Filter YearBuilt invalid values
if 'YearBuilt' in df_clean.columns:
    df_clean = df_clean[(df_clean['YearBuilt'].isnull()) | 
                        ((df_clean['YearBuilt'] >= 1800) & (df_clean['YearBuilt'] <= 2026))]

print(f"\nDataset shape before cleaning: {df.shape}")
print(f"Dataset shape after filtering anomalies: {df_clean.shape}")
# ==============================================================================
# 4. TRAIN-TEST SPLIT (PREVENTING DATA LEAKAGE)
# ==============================================================================
X = df_clean.drop(columns=['Price'])
y = df_clean['Price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining set shape: {X_train.shape}, Test set shape: {X_test.shape}")

# ==============================================================================
# 5. FEATURE ENGINEERING & IMPUTATION (STRICTLY ON TRAIN DATA)
# ==============================================================================
X_train = X_train.copy()
X_test = X_test.copy()

# 5.1 Extract time-based features
if 'Date' in X_train.columns:
    for data in [X_train, X_test]:
        data['Date'] = pd.to_datetime(data['Date'], dayfirst=True)
        data['SaleYear'] = data['Date'].dt.year
        data['SaleMonth'] = data['Date'].dt.month
        
        if 'YearBuilt' in data.columns:
            data['BuildingAge'] = data['SaleYear'] - data['YearBuilt']
        
        data.drop(columns=['Date'], inplace=True)

# ==============================================================================
# 5.2 Landsize imputation by subgroup medians (Suburb + Type)
# ==============================================================================
# 1. Compute medians strictly on Train
landsize_medians = (
    X_train.groupby(['Suburb', 'Type'])['Landsize']
    .median()
    .reset_index()
    .rename(columns={'Landsize': 'Landsize_median'})
)

# 2. Merge medians to Train and Test
X_train = X_train.merge(landsize_medians, on=['Suburb', 'Type'], how='left')
X_train['Landsize'] = X_train['Landsize'].fillna(X_train['Landsize_median'])
X_train.drop(columns=['Landsize_median'], inplace=True)

X_test = X_test.merge(landsize_medians, on=['Suburb', 'Type'], how='left')
X_test['Landsize'] = X_test['Landsize'].fillna(X_test['Landsize_median'])
X_test.drop(columns=['Landsize_median'], inplace=True)

# 3. Fallback to global train median for unseen Suburb+Type combinations or remaining NaNs
global_landsize = X_train['Landsize'].median()
X_train['Landsize'] = X_train['Landsize'].fillna(global_landsize)
X_test['Landsize'] = X_test['Landsize'].fillna(global_landsize)
# 5.3 Numerical feature imputation via Train medians
num_cols_to_impute = ['BuildingArea', 'BuildingAge', 'Car']
for col in num_cols_to_impute:
    if col in X_train.columns:
        med = X_train[col].median()
        X_train[col] = X_train[col].fillna(med)
        X_test[col] = X_test[col].fillna(med)

# 5.4 Categorical feature imputation via Train mode
if 'CouncilArea' in X_train.columns:
    mode_council = X_train['CouncilArea'].mode()[0]
    X_train['CouncilArea'] = X_train['CouncilArea'].fillna(mode_council)
    X_test['CouncilArea'] = X_test['CouncilArea'].fillna(mode_council)

print("\n--- Remaining Missing Values in X_train ---")
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

# ==============================================================================
# 6. FEATURE PREPARATION FOR MODELING
# ==============================================================================
# Drop non-predictive text identifier columns
drop_cols = ['Address']
X_train = X_train.drop(columns=[c for c in drop_cols if c in X_train.columns])
X_test = X_test.drop(columns=[c for c in drop_cols if c in X_test.columns])

# Identify numerical and categorical columns
num_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"\nNumerical Features ({len(num_features)}): {num_features}")
print(f"Categorical Features ({len(cat_features)}): {cat_features}")

# ==============================================================================
# 7. MODELING & COMPARATIVE EVALUATION
# ==============================================================================
from sklearn.impute import SimpleImputer

results = []

def evaluate_predictions(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    results.append({
        'Model': model_name,
        'MAE': mae,
        'R2': r2
    })
    print(f"[{model_name}] MAE: {mae:,.2f} | R2: {r2:,.2f}")

# ------------------------------------------------------------------------------
# 7.1 Baseline Model: Linear Regression (Ridge)
# ----------------================----------------------------------------------
# Numeric pipeline: Impute remaining NaNs with median -> Scale
num_transformer_lr = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: Impute NaNs with most frequent -> OneHotEncode
cat_transformer_lr = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor_lr = ColumnTransformer(
    transformers=[
        ('num', num_transformer_lr, num_features),
        ('cat', cat_transformer_lr, cat_features)
    ]
)

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_lr),
    ('regressor', Ridge(alpha=1.0))
])

lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)
evaluate_predictions('Linear Regression (Ridge)', y_test, lr_preds)


# ------------------------------------------------------------------------------
# 7.2 Ensemble Model: Random Forest Regressor
# ------------------------------------------------------------------------------
num_transformer_rf = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor_rf = ColumnTransformer(
    transformers=[
        ('num', num_transformer_rf, num_features),
        ('cat', cat_transformer_lr, cat_features)
    ]
)

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_rf),
    ('regressor', RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict(X_test)
evaluate_predictions('Random Forest Regressor', y_test, rf_preds)


# ------------------------------------------------------------------------------
# 7.3 Gradient Boosting Model: LightGBM Regressor
# ------------------------------------------------------------------------------
X_train_lgb = X_train.copy()
X_test_lgb = X_test.copy()

# Convert categorical features to category dtype for LightGBM native support
for c in cat_features:
    X_train_lgb[c] = X_train_lgb[c].astype('category')
    X_test_lgb[c] = X_test_lgb[c].astype('category')

lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

lgb_model.fit(X_train_lgb, y_train)
lgb_preds = lgb_model.predict(X_test_lgb)
evaluate_predictions('LightGBM Regressor', y_test, lgb_preds)

--- First 5 rows of the dataset ---


,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,68 Studley St,2,h,NaN,SS,Jellis,3/09/2016,2.5,3067.0,2.0,1.0,1.0,126.0,NaN,NaN,Yarra City Council,-37.8014,144.9958,Northern Metropolitan,4019.0
1,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,2.0,1.0,1.0,202.0,NaN,NaN,Yarra City Council,-37.7996,144.9984,Northern Metropolitan,4019.0
2,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,2.0,1.0,0.0,156.0,79.0,1900.0,Yarra City Council,-37.8079,144.9934,Northern Metropolitan,4019.0
3,Abbotsford,18/659 Victoria St,3,u,NaN,VB,Rounds,4/02/2016,2.5,3067.0,3.0,2.0,1.0,0.0,NaN,NaN,Yarra City Council,-37.8114,145.0116,Northern Metropolitan,4019.0
4,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,3.0,2.0,0.0,134.0,150.0,1900.0,Yarra City Council,-37.8093,144.9944,Northern Metropolitan,4019.0



--- Dataset Info & Data Types ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34857 entries, 0 to 34856
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         34857 non-null  object 
 1   Address        34857 non-null  object 
 2   Rooms          34857 non-null  int64  
 3   Type           34857 non-null  object 
 4   Price          27247 non-null  float64
 5   Method         34857 non-null  object 
 6   SellerG        34857 non-null  object 
 7   Date           34857 non-null  object 
 8   Distance       34856 non-null  float64
 9   Postcode       34856 non-null  float64
 10  Bedroom2       26640 non-null  float64
 11  Bathroom       26631 non-null  float64
 12  Car            26129 non-null  float64
 13  Landsize       23047 non-null  float64
 14  BuildingArea   13742 non-null  float64
 15  YearBuilt      15551 non-null  float64
 16  CouncilArea    34854 non-null  object 
 17  Lattitude      

,Rooms,Price,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,34857.000000,2.724700e+04,34856.000000,34856.000000,26640.000000,26631.000000,26129.000000,23047.000000,13742.00000,15551.000000,26881.000000,26881.000000,34854.000000
mean,3.031012,1.050173e+06,11.184929,3116.062859,3.084647,1.624798,1.728845,593.598993,160.25640,1965.289885,-37.810634,145.001851,7572.888306
std,0.969933,6.414671e+05,6.788892,109.023903,0.980690,0.724212,1.010771,3398.841946,401.26706,37.328178,0.090279,0.120169,4428.090313
min,1.000000,8.500000e+04,0.000000,3000.000000,0.000000,0.000000,0.000000,0.000000,0.00000,1196.000000,-38.190430,144.423790,83.000000
25%,2.000000,6.350000e+05,6.400000,3051.000000,2.000000,1.000000,1.000000,224.000000,102.00000,1940.000000,-37.862950,144.933500,4385.000000
50%,3.000000,8.700000e+05,10.300000,3103.000000,3.000000,2.000000,2.000000,521.000000,136.00000,1970.000000,-37.807600,145.007800,6763.000000
75%,4.000000,1.295000e+06,14.000000,3156.000000,4.000000,2.000000,2.000000,670.000000,188.00000,2000.000000,-37.754100,145.071900,10412.000000
max,16.000000,1.120000e+07,48.100000,3978.000000,30.000000,12.000000,26.000000,433014.000000,44515.00000,2106.000000,-37.390200,145.526350,21650.000000



--- Missing Values Count per Column ---


Price             7610
Distance             1
Postcode             1
Bedroom2          8217
Bathroom          8226
Car               8728
Landsize         11810
BuildingArea     21115
YearBuilt        19306
CouncilArea          3
Lattitude         7976
Longtitude        7976
Regionname           3
Propertycount        3
dtype: int64


Dataset shape before cleaning: (34857, 21)
Dataset shape after filtering anomalies: (27111, 21)

Training set shape: (21688, 20), Test set shape: (5423, 20)

--- Remaining Missing Values in X_train ---
Distance             1
Postcode             1
Bedroom2          5129
Bathroom          5133
YearBuilt        12054
Lattitude         4983
Longtitude        4983
Regionname           3
Propertycount        3
dtype: int64

Numerical Features (13): ['Rooms', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'YearBuilt', 'Lattitude', 'Longtitude', 'Propertycount', 'BuildingAge']
Categorical Features (6): ['Suburb', 'Type', 'Method', 'SellerG', 'CouncilArea', 'Regionname']
[Linear Regression (Ridge)] MAE: 231,653.37 | R2: 0.68
[Random Forest Regressor] MAE: 170,765.46 | R2: 0.79
[LightGBM Regressor] MAE: 163,934.27 | R2: 0.82
